In [1]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2026-03-09T19:06:55.691788+11:00

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.6.0

Compiler    : Clang 16.0.0 (clang-1600.0.26.6)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit




### Extract more sentences from BNC

This notebook extracts more sentences from BNC for previously identified sentences that contain literal senses of the target words. We extract 15 previous sentences for written texts and 30 for spoken texts because sentences for the latter are relatively short.

In [31]:
import os
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

animal_path   = Path(project_folder) / "data" / "wsd" / "animals_wsd.csv"
flea_path     = Path(project_folder) / "data" / "wsd" / "animals_wsd_flea.csv"
artifact_path = Path(project_folder) / "data" / "wsd" / "artifacts_wsd.csv"

animals_other = pd.read_csv(animal_path)
animals_flea  = pd.read_csv(flea_path)
animals = pd.concat([animals_other, animals_flea], ignore_index=True).assign(domain="animal")
artifacts = pd.read_csv(artifact_path).assign(domain="artifact")

# filter sentences that contain literal sentences
animals = animals[(animals["label"] == "ANIMAL") & (animals["confidence"] >= 0.95)]
artifacts = artifacts[(artifacts["label"] == "ARTIFACT") & (artifacts["confidence"] >= 0.95)]

df = pd.concat([animals, artifacts], ignore_index=True)

df["sentence_id"] = (
    pd.to_numeric(df["sentence_id"], errors="coerce")
      .astype("Int64")
      .astype(str)
)
df["file_path"] = df["file_path"].astype(str).str.strip()

# there are some missing values for sentence ID
df["sentence_id"] = df["sentence_id"].replace("<NA>", pd.NA)
df.loc[df["sentence_id"].isna(), ["file_path", "sentence_id", "sentence", "target_word"]]

,file_path,sentence_id,sentence,target_word
5401,H81,<NA>,"But though this was so, Lord Coke reports that...",horse
12009,KDP,<NA>,A **shoe**.,shoe


In [32]:
# replace those missing values
df.loc[5401, "sentence_id"] = "1303_1"
df.loc[12009, "sentence_id"] = "2834_1"
# no missing values should be found
assert df["sentence_id"].notna().all()
assert df["file_path"].notna().all()

In [33]:
texts_root  = Path(project_folder) / "rawdata" / "BNC" / "download" / "Texts"
out_csv     = Path(project_folder) / "data" / "wsd" / "all_context.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)

spoken_types = {"OTHERSP", "CONVRSN"} 

def localname(tag: str) -> str:
    """Return tag localname without namespace, lowercased."""
    return tag.rsplit("}", 1)[-1].lower() if tag else ""

def sentence_surface_from_s_el(s_el: ET.Element) -> str:
    """
    Reconstruct sentence surface using the same concatenation method as before:
    concatenate the raw .text of <w> and <c> (including nested ones), then strip.
    """
    tokens = []
    for child in list(s_el):
        tag = localname(child.tag)
        if tag in {"w", "c"}:
            tokens.append(child.text or "")
        else:
            for g in child.iter():
                gtag = localname(g.tag)
                if gtag in {"w", "c"}:
                    tokens.append(g.text or "")
    return "".join(tokens).strip()

def find_xml_by_stem(stem: str, cache: dict) -> Path | None:
    """Find the first XML file under texts_root whose stem matches `stem`."""
    if stem in cache:
        return cache[stem]
    for p in texts_root.rglob(f"{stem}.xml"):
        cache[stem] = p
        return p
    cache[stem] = None
    return None

def get_doc_type(root: ET.Element) -> str | None:
    """Prefer wtext@type, fallback to stext@type."""
    wtext = None
    stext = None
    for el in root.iter():
        name = localname(el.tag)
        if name == "wtext" and wtext is None:
            wtext = el
        elif name == "stext" and stext is None:
            stext = el
        if wtext is not None and stext is not None:
            break

    if wtext is not None and wtext.get("type"):
        return wtext.get("type")
    if stext is not None and stext.get("type"):
        return stext.get("type")
    return None

def extract_prev_k_sentences_from_root(root: ET.Element, sentence_id: str, k: int) -> str | None:
    """
    Return up to k previous sentences in linear document order (ignores <p>),
    concatenated with single spaces.
    """
    s_els = [el for el in root.iter() if localname(el.tag) == "s"]
    if not s_els:
        return None

    target_id = str(sentence_id).strip()
    ids = [(s.get("n") or "").strip() for s in s_els]

    try:
        idx = ids.index(target_id)
    except ValueError:
        return None

    start = max(0, idx - k)
    prev_surfs = []
    for s in s_els[start:idx]:
        surf = sentence_surface_from_s_el(s)
        if surf:
            prev_surfs.append(surf)

    return " ".join(prev_surfs).strip()

# for each unique combination of file path and sentence ID, we will extract previous sentences
targets = df[["file_path", "sentence_id"]].dropna().drop_duplicates()

xml_cache = {}
ctx_rows = []

for file_path, sent_id in targets.itertuples(index=False):
    xml_path = find_xml_by_stem(str(file_path), xml_cache)

    if xml_path is None:
        ctx_rows.append({
            "file_path": file_path,
            "sentence_id": sent_id,
            "doc_type": None,
            "k_used": None,
            "previous_sentences": None,
        })
        continue

    try:
        root = ET.parse(str(xml_path)).getroot()
    except ET.ParseError:
        ctx_rows.append({
            "file_path": file_path,
            "sentence_id": sent_id,
            "doc_type": None,
            "k_used": None,
            "previous_sentences": None,
        })
        continue

    doc_type = get_doc_type(root)
    k = 30 if (doc_type in spoken_types) else 15 # extract 15 if written and 30 if spoken

    prev_ctx = extract_prev_k_sentences_from_root(root, sent_id, k=k)

    ctx_rows.append({
        "file_path": file_path,
        "sentence_id": sent_id,
        "doc_type": doc_type,
        "k_used": k,
        "previous_sentences": prev_ctx,
    })

df = df.drop(columns=["previous_sentence"], errors="ignore")

ctx_df = pd.DataFrame(ctx_rows, columns=[
    "file_path", "sentence_id", "k_used", "previous_sentences"
])

df_out = df.merge(ctx_df, on=["file_path", "sentence_id"], how="left")

df_out.to_csv(out_csv, index=False)
print(f"Saved {len(df_out):,} rows to {out_csv}")

Saved 17,730 rows to /Users/temuulenkh/Documents/github/features/data/wsd/all_context.csv


For efficiency when estimating contextual probabilities using T5 model, we sort the input sentences by token size so that inputs with similar size are batched together and longer token sentences are at the end in case any issue arises.

In [2]:
import os
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer
from tqdm.auto import tqdm

project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

all_path = Path(project_folder) / "data" / "wsd" / "all_context.csv"
out_sorted_path = Path(project_folder) / "data" / "wsd" / "all_context_sorted.csv"

df = pd.read_csv(all_path)

def build_input(df):
    return (
        df["previous_sentences"].fillna("") + " " +
        df["sentence"].fillna("") + " " +
        df["next_sentence"].fillna("")
    ).str.strip()

df = df.assign(input=build_input(df))[
    ["domain", "row_id", "doc_type", "input", "target_word"]
]

print("Rows:", len(df))

MODEL_NAME = "google-t5/t5-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def token_len(text: str) -> int:
    return len(tokenizer.encode(str(text), add_special_tokens=False))

tqdm.pandas()
df["n_tokens"] = df["input"].progress_apply(token_len)

df_sorted = df.sort_values("n_tokens", ascending=True).reset_index(drop=True)
print(df_sorted["n_tokens"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

df_sorted.to_csv(out_sorted_path, index=False)
print(f"Saved {len(df_sorted):,} rows to {out_sorted_path}")

Rows: 17730


  0%|          | 0/17730 [00:00<?, ?it/s]

count    17730.000000
mean       447.899549
std        240.550196
min         19.000000
50%        403.000000
90%        692.000000
95%        852.000000
99%       1353.130000
max       6334.000000
Name: n_tokens, dtype: float64
Saved 17,730 rows to /Users/temuulenkh/Documents/github/features/data/wsd/all_context_sorted.csv
